In [2]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
from pyspark.sql.window import Window 

spark = SparkSession.builder \
    .appName("MOOCCubeX_Preprocessing") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN") 
print("Spark version:", spark.version)

Spark version: 4.1.1


In [6]:
# --- exercise-problem.txt ---
exercise_problem_raw = spark.read.csv("C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/relations/exercise-problem.csv", header=True).cache()
print("=== exercise-problem sample ===")
exercise_problem_raw.show(3, truncate=False)

=== exercise-problem sample ===
+-----------+----------+
|exercise_id|problem_id|
+-----------+----------+
|Ex_143     |Pm_1      |
|Ex_143     |Pm_2      |
|Ex_147     |Pm_3      |
+-----------+----------+
only showing top 3 rows


In [7]:
# --- course.json ---
course_raw = spark.read.json("C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/entities/course.json")
print("=== course schema ===")
course_raw.show(3)


=== course schema ===
+-------------------------------------+------------------------------+--------+--------------------------+-------------+--------------------+
|                                about|                         field|      id|                      name|prerequisites|            resource|
+-------------------------------------+------------------------------+--------+--------------------------+-------------+--------------------+
|通过老师导读，同学们可深入这一经典...|        [历史学, 中国语言文学]|C_584313|          《资治通鉴》导读|             |[{1.1.1, V_849, [...|
|本课程是理工科的一门数学基础课，系...|[应用经济学, 数学, 物理学, ...|C_584329|微积分——极限理论与一元函数|             |[{1.1.1, V_1350, ...|
|掌握基本的摄影技能，了解图片新闻的...|          [艺术学, 新闻传播学]|C_584381|                  新闻摄影|             |[{1.1.1, V_1800, ...|
+-------------------------------------+------------------------------+--------+--------------------------+-------------+--------------------+
only showing top 3 rows


In [8]:

# --- user-video.json ---
user_video_raw = spark.read.json("C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data/relations/user-video.json").cache()
print("=== user-video schema ===")
user_video_raw.show(3, truncate=False)

=== user-video schema ===
+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|_corrupt_record|seq                                                                                                                                                                                                                                                                                                                                                                                                                      

==============================PREPARATION=================================

In [9]:

test = course_raw.filter(
    (col("prerequisites").isNotNull()) &
    (trim(col("prerequisites")) != "")
)

test.createOrReplaceTempView("my_table")
spark.sql("select distinct prerequisites from my_table").show()

+--------------------------------------+
|                         prerequisites|
+--------------------------------------+
|                          高中化学知识|
| 完成《网络技术与应用》课程学习或同...|
| 程序设计初学者。无先修课程要求，如...|
|    具备一定的高数和高中物理知识储备。|
|    计算机基础知识和汉语知识。（ Ba...|
|              对戏曲艺术有一定兴趣爱好|
|                                 C语言|
|理论力学、机械制图、机械原理、机械设计|
|                        不需要先修知识|
|   罗素《西方哲学史》\n夏正林《思维...|
|                              高等数学|
|                      化学、生物化学等|
|  高中以上文化水平皆可选择此门课程学习|
|    材料力学、结构力学、混凝土结构原理|
|高等数学、线性代数、理论力学、材料力学|
|              无需先修知识和先修课程。|
|            无需预备知识 No pre - c...|
| 没有先修课要求，任何喜欢礼仪的小伙...|
| 如能先读一读老子的《道德经》，会有...|
|               先修课程为《古代汉语...|
+--------------------------------------+
only showing top 20 rows


In [17]:
path_concept = "C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\relations\\concept-course.csv"

concept_course = spark.read.csv(path_concept, header=True)
concept_course.select("term", "course_id").dropDuplicates().count()


451078

In [ ]:
#join concept
course_concept_origin = spark.read.csv("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\output\\course_concept_origin.csv", header="true")
#schema (course_id, concept)
course_concept_trans = spark.read.csv("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\output\\course_concept_trans.csv", header="true")
#schema (concept,name_en)


course_concept_transformed = course_concept_origin.join(broadcast(course_concept_trans),   # tối ưu vì bảng dịch nhỏ hơn
    on="concept",
    how="left")\
        .withColumnRenamed("name_en", "term")\
        .drop("concept")



In [9]:
DATA_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data"
concept_course_df = spark.read.csv(f"{DATA_PATH}/relations/concept-course.csv", header=True)
# Group concepts by course
course_concepts = concept_course_df.groupBy("course_id").agg(
    collect_list("term").alias("concepts")
)

# Count concepts per course
course_concepts = course_concepts.withColumn("num_concepts", size("concepts"))

print(f"Courses with concepts: {course_concepts.count():,}")

# Statistics
print("\nConcepts per course statistics:")
course_concepts.select("num_concepts").summary().show()


Courses with concepts: 887

Concepts per course statistics:
+-------+------------------+
|summary|      num_concepts|
+-------+------------------+
|  count|               887|
|   mean|  508.543404735062|
| stddev|454.52013005124786|
|    min|                 1|
|    25%|               186|
|    50%|               402|
|    75%|               727|
|    max|              3515|
+-------+------------------+



In [9]:
course_concepts.show(5)

+---------+--------------------------------+------------+
|course_id|                        concepts|num_concepts|
+---------+--------------------------------+------------+
| C_883345| [K_比较器_计算机科学与技术, ...|        1484|
| C_707083|[K_传送信号_计算机科学与技术,...|        2051|
| C_696855|  [K_程序_计算机科学与技术, K...|        2091|
| C_697075|   [K_一偶函数_应用经济学, K_...|        1007|
| C_584329|   [K_一偶函数_应用经济学, K_...|        1007|
+---------+--------------------------------+------------+
only showing top 5 rows


In [6]:
DATA_PATH = "C:/Hanu/year3/year3_semester2/BDM/final_project/MOOCCubeX/data"
concept_course_df_1 = spark.read.csv(f"{DATA_PATH}/relations/concept-course_trans.csv", header=True)
# Group concepts by course
course_concepts_1 = concept_course_df_1.groupBy("course_id").agg(
    collect_list("term").alias("concepts")
)

# Count concepts per course
course_concepts_1 = course_concepts_1.withColumn("num_concepts", size("concepts"))

print(f"Courses with concepts: {course_concepts_1.count():,}")

# Statistics
print("\nConcepts per course statistics:")
course_concepts_1.select("num_concepts").summary().show()


Courses with concepts: 887

Concepts per course statistics:
+-------+------------------+
|summary|      num_concepts|
+-------+------------------+
|  count|               887|
|   mean|  508.543404735062|
| stddev|454.52013005124786|
|    min|                 1|
|    25%|               186|
|    50%|               402|
|    75%|               727|
|    max|              3515|
+-------+------------------+



In [11]:
course_concepts_1.createOrReplaceTempView("test1")
res = spark.sql("select * from test1 where course_id = 'C_1755888'")
res.show()

+---------+--------------------+------------+
|course_id|            concepts|num_concepts|
+---------+--------------------+------------+
|C_1755888|[K_Position_Arts,...|          36|
+---------+--------------------+------------+



In [12]:
course_concepts.createOrReplaceTempView("test2")
res_1 = spark.sql("select * from test2 where course_id = 'C_1755888'")
res_1.show()

+---------+-----------------------------+------------+
|course_id|                     concepts|num_concepts|
+---------+-----------------------------+------------+
|C_1755888|[K_基本形象_艺术学, K_遒劲...|          36|
+---------+-----------------------------+------------+



In [ ]:
course_concept_transformed.coalesce(1).write.option("header","true").mode('overwrite').csv("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\output\\course_concept_transformed.csv")

In [ ]:
course_concept_origin = spark.read.csv("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\relation\\course_concept.csv", header="true")
#schema (course_id, concept)
course_concept_trans = spark.read.csv("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\output\\course_concept_trans.csv", header="true")

In [22]:
user_df = spark.read.json("C:\\Hanu\\year3\\year3_semester2\\BDM\\final_project\\MOOCCubeX\\data\\entities\\user.json")
user_df.createOrReplaceTempView("test3")
res = spark.sql("SELECT id, size(course_order) AS total_order FROM test3 WHERE size(course_order) > 3") 
res.count()

57994